# Hospital Admissions ETL Pipeline
Extract, transform and load real hospital data (MIMIC-IV demo) using PySpark and Neon PostgreSQL.

## 1. Setup

In [1]:
%pip -q install pyspark psycopg2-binary

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("hospital_etl").master("local[*]").getOrCreate()
print("Spark version:", spark.version)

Spark version: 4.0.4


## 2. Extract
The data is the MIMIC-IV Clinical Database Demo from PhysioNet. It is real hospital data (100 patients, identifiers removed). Four tables are used: patients, admissions, diagnoses_icd and d_icd_diagnoses (the names of the diagnosis codes).

In [3]:
import os
import subprocess

base = "https://physionet.org/files/mimic-iv-demo/2.2/hosp/"
names = ["patients", "admissions", "diagnoses_icd", "d_icd_diagnoses"]

os.makedirs("raw", exist_ok=True)
for n in names:
    subprocess.run(["wget", "-q", "-O", "raw/" + n + ".csv.gz", base + n + ".csv.gz?download"], check=True)

print(os.listdir("raw"))

['admissions.csv.gz', 'diagnoses_icd.csv.gz', 'd_icd_diagnoses.csv.gz', 'patients.csv.gz']


In [4]:
pat_raw = spark.read.option("header", True).csv("raw/patients.csv.gz")
adm_raw = spark.read.option("header", True).csv("raw/admissions.csv.gz")
dx_raw = spark.read.option("header", True).csv("raw/diagnoses_icd.csv.gz")
icd_raw = spark.read.option("header", True).csv("raw/d_icd_diagnoses.csv.gz")

pat_n = pat_raw.count()
adm_n = adm_raw.count()
dx_n = dx_raw.count()

print("patients:", pat_n, "rows")
print("admissions:", adm_n, "rows")
print("diagnoses:", dx_n, "rows")
print("icd names:", icd_raw.count(), "rows")

adm_raw.show(5, truncate=False)

patients: 100 rows
admissions: 275 rows
diagnoses: 4506 rows
icd names: 109775 rows
+----------+--------+-------------------+-------------------+-------------------+--------------+-----------------+----------------------+------------------------+---------+--------+--------------+------------------------------+-------------------+-------------------+--------------------+
|subject_id|hadm_id |admittime          |dischtime          |deathtime          |admission_type|admit_provider_id|admission_location    |discharge_location      |insurance|language|marital_status|race                          |edregtime          |edouttime          |hospital_expire_flag|
+----------+--------+-------------------+-------------------+-------------------+--------------+-----------------+----------------------+------------------------+---------+--------+--------------+------------------------------+-------------------+-------------------+--------------------+
|10004235  |24181354|2196-02-24 14:38:00|2196-03-

## 3. Transform
### 3.1 Clean and set data types

In [5]:
from pyspark.sql import functions as F

def trim_all(d):
    for c in d.columns:
        d = d.withColumn(c, F.trim(F.col(c)))
    return d

pat = trim_all(pat_raw)
pat = (pat.withColumn("subject_id", F.col("subject_id").cast("int"))
          .withColumn("gender", F.upper(F.col("gender")))
          .withColumn("anchor_age", F.col("anchor_age").cast("int"))
          .withColumn("anchor_year", F.col("anchor_year").cast("int"))
          .withColumn("dod", F.to_timestamp("dod")))

adm = trim_all(adm_raw)
adm = (adm.withColumn("subject_id", F.col("subject_id").cast("int"))
          .withColumn("hadm_id", F.col("hadm_id").cast("int"))
          .withColumn("admittime", F.to_timestamp("admittime"))
          .withColumn("dischtime", F.to_timestamp("dischtime"))
          .withColumn("hospital_expire_flag", F.col("hospital_expire_flag").cast("int")))

# missing values and ? in these columns become UNKNOWN
txt = ["admission_location", "discharge_location", "insurance", "language", "marital_status", "race"]
for c in txt:
    adm = adm.withColumn(c, F.when(F.col(c).isNull() | (F.col(c) == "") | (F.col(c) == "?"), "UNKNOWN").otherwise(F.col(c)))

adm = adm.select("subject_id", "hadm_id", "admittime", "dischtime", "admission_type", *txt, "hospital_expire_flag")

dx = trim_all(dx_raw)
dx = (dx.withColumn("hadm_id", F.col("hadm_id").cast("int"))
        .withColumn("seq_num", F.col("seq_num").cast("int"))
        .withColumn("icd_version", F.col("icd_version").cast("int")))

icd = trim_all(icd_raw).withColumn("icd_version", F.col("icd_version").cast("int"))

adm.printSchema()

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- admittime: timestamp (nullable = true)
 |-- dischtime: timestamp (nullable = true)
 |-- admission_type: string (nullable = true)
 |-- admission_location: string (nullable = true)
 |-- discharge_location: string (nullable = true)
 |-- insurance: string (nullable = true)
 |-- language: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- race: string (nullable = true)
 |-- hospital_expire_flag: integer (nullable = true)



### 3.2 Data quality checks

In [6]:
print("Duplicate patient ids:", pat.count() - pat.select("subject_id").distinct().count())
print("Duplicate admission ids:", adm.count() - adm.select("hadm_id").distinct().count())
print("Missing admit time:", adm.filter(F.col("admittime").isNull()).count())
print("Discharge before admit:", adm.filter(F.col("dischtime") < F.col("admittime")).count())
print("Admissions without a patient:", adm.join(pat, "subject_id", "left_anti").count())
print("Diagnoses without an admission:", dx.join(adm, "hadm_id", "left_anti").count())

print("Missing values left in admissions:")
adm.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in adm.columns]).show()

Duplicate patient ids: 0
Duplicate admission ids: 0
Missing admit time: 0
Discharge before admit: 0
Admissions without a patient: 0
Diagnoses without an admission: 0
Missing values left in admissions:
+----------+-------+---------+---------+--------------+------------------+------------------+---------+--------+--------------+----+--------------------+
|subject_id|hadm_id|admittime|dischtime|admission_type|admission_location|discharge_location|insurance|language|marital_status|race|hospital_expire_flag|
+----------+-------+---------+---------+--------------+------------------+------------------+---------+--------+--------------+----+--------------------+
|         0|      0|        0|        0|             0|                 0|                 0|        0|       0|             0|   0|                   0|
+----------+-------+---------+---------+--------------+------------------+------------------+---------+--------+--------------+----+--------------------+



### 3.3 Remove bad rows
Rows that fail a check are left out of the load. The counts are saved in the etl_log table later.

In [7]:
pat_ok = pat.dropna(subset=["subject_id"]).dropDuplicates(["subject_id"])

bad = (F.col("hadm_id").isNull()
       | F.col("admittime").isNull()
       | F.coalesce(F.col("dischtime") < F.col("admittime"), F.lit(False)))

adm_ok = (adm.filter(~bad)
             .dropDuplicates(["hadm_id"])
             .join(pat_ok.select("subject_id"), "subject_id", "left_semi"))

icd_ok = icd.dropDuplicates(["icd_code", "icd_version"]).select("icd_code", "icd_version", F.col("long_title").alias("dx_title"))

dx_ok = (dx.dropna(subset=["hadm_id", "seq_num", "icd_code"])
           .dropDuplicates(["hadm_id", "seq_num"])
           .join(adm_ok.select("hadm_id"), "hadm_id", "left_semi")
           .join(icd_ok, ["icd_code", "icd_version"], "left")
           .withColumn("dx_title", F.coalesce(F.col("dx_title"), F.lit("UNKNOWN"))))

print("patients kept:", pat_ok.count(), "of", pat_n)
print("admissions kept:", adm_ok.count(), "of", adm_n)
print("diagnoses kept:", dx_ok.count(), "of", dx_n)

patients kept: 100 of 100
admissions kept: 275 of 275
diagnoses kept: 4506 of 4506


### 3.4 Add new columns
Length of stay, age at admission, number of diagnoses and the main (first) diagnosis for each admission.
The dates in this data are shifted on purpose to hide identities, so the age is worked out from anchor_age and anchor_year.

In [8]:
first_dx = (dx_ok.filter(F.col("seq_num") == 1)
                 .select("hadm_id", F.col("icd_code").alias("primary_dx_code"), F.col("dx_title").alias("primary_dx_title")))

n_dx = dx_ok.groupBy("hadm_id").agg(F.count("*").alias("num_diagnoses"))

adm_out = (adm_ok.join(pat_ok.select("subject_id", "anchor_age", "anchor_year"), "subject_id")
                 .withColumn("los_days", F.round((F.unix_timestamp("dischtime") - F.unix_timestamp("admittime")) / 86400, 2))
                 .withColumn("age_at_admit", F.col("anchor_age") + F.year("admittime") - F.col("anchor_year"))
                 .join(n_dx, "hadm_id", "left")
                 .join(first_dx, "hadm_id", "left")
                 .withColumn("num_diagnoses", F.coalesce(F.col("num_diagnoses"), F.lit(0)))
                 .drop("anchor_age", "anchor_year"))

pat_out = pat_ok.select("subject_id", "gender", "anchor_age", "anchor_year", "anchor_year_group", "dod")
dx_out = dx_ok.select("hadm_id", "seq_num", "icd_code", "icd_version", "dx_title")

adm_out.select("hadm_id", "admission_type", "los_days", "age_at_admit", "num_diagnoses", "primary_dx_title").show(5, truncate=False)

+--------+-----------------+--------+------------+-------------+--------------------------------------------------+
|hadm_id |admission_type   |los_days|age_at_admit|num_diagnoses|primary_dx_title                                  |
+--------+-----------------+--------+------------+-------------+--------------------------------------------------+
|20044587|ELECTIVE         |5.29    |70          |8            |Coronary atherosclerosis of native coronary artery|
|20093566|OBSERVATION ADMIT|4.07    |51          |32           |Alcoholic hepatic failure without coma            |
|20192635|EW EMER.         |3.17    |80          |18           |Poisoning by insulins and antidiabetic agents     |
|20199380|OBSERVATION ADMIT|4.67    |40          |5            |Acute embolism and thrombosis of left iliac vein  |
|20214994|URGENT           |23.24   |75          |36           |Malignant neoplasm of anus, unspecified site      |
+--------+-----------------+--------+------------+-------------+--------

## 4. Load
The tables are loaded into Neon PostgreSQL. The connection string is saved in Colab Secrets with the name `NEON_DATABASE_URL`.

In [16]:
import psycopg2
from google.colab import userdata

conn = psycopg2.connect(userdata.get("NEON_DATABASE_URL"))
cur = conn.cursor()
print("Connected to PostgreSQL")

Connected to PostgreSQL


In [17]:
cur.execute("""
CREATE TABLE IF NOT EXISTS patients (
    subject_id INTEGER PRIMARY KEY,
    gender VARCHAR(1),
    anchor_age INTEGER,
    anchor_year INTEGER,
    anchor_year_group TEXT,
    dod TIMESTAMP
);

CREATE TABLE IF NOT EXISTS admissions (
    hadm_id INTEGER PRIMARY KEY,
    subject_id INTEGER NOT NULL REFERENCES patients (subject_id),
    admittime TIMESTAMP NOT NULL,
    dischtime TIMESTAMP,
    admission_type TEXT,
    admission_location TEXT,
    discharge_location TEXT,
    insurance TEXT,
    language TEXT,
    marital_status TEXT,
    race TEXT,
    hospital_expire_flag SMALLINT,
    los_days NUMERIC(8, 2),
    age_at_admit INTEGER,
    num_diagnoses INTEGER,
    primary_dx_code TEXT,
    primary_dx_title TEXT
);

CREATE TABLE IF NOT EXISTS diagnoses (
    hadm_id INTEGER NOT NULL REFERENCES admissions (hadm_id),
    seq_num INTEGER NOT NULL,
    icd_code TEXT,
    icd_version SMALLINT,
    dx_title TEXT,
    PRIMARY KEY (hadm_id, seq_num)
);

CREATE TABLE IF NOT EXISTS etl_log (
    id SERIAL PRIMARY KEY,
    run_time TIMESTAMP DEFAULT NOW(),
    table_name TEXT,
    rows_read INTEGER,
    rows_loaded INTEGER,
    rows_rejected INTEGER
);
""")
conn.commit()
print("Tables ready")

Tables ready


In [18]:
from psycopg2.extras import execute_values

def load(df, table, cols):
    rows = [tuple(r) for r in df.select(*cols).collect()]
    q = "INSERT INTO " + table + " (" + ", ".join(cols) + ") VALUES %s"
    execute_values(cur, q, rows, page_size=500)
    return len(rows)

# order matters because of the foreign keys
jobs = [
    ("patients", pat_out, pat_n),
    ("admissions", adm_out, adm_n),
    ("diagnoses", dx_out, dx_n),
]

cur.execute("TRUNCATE diagnoses, admissions, patients")

for table, df_out, n_in in jobs:
    n_out = load(df_out, table, df_out.columns)
    cur.execute(
        "INSERT INTO etl_log (table_name, rows_read, rows_loaded, rows_rejected) VALUES (%s, %s, %s, %s)",
        (table, n_in, n_out, n_in - n_out)
    )
    print(table, "->", n_out, "rows loaded,", n_in - n_out, "rejected")

conn.commit()

patients -> 100 rows loaded, 0 rejected
admissions -> 275 rows loaded, 0 rejected
diagnoses -> 4506 rows loaded, 0 rejected


## 5. Analytics
SQL queries on the loaded tables.

In [19]:
queries = {
    "Admissions and average stay by admission type": """
SELECT
    admission_type,
    COUNT(*) AS total_admissions,
    ROUND(AVG(los_days), 1) AS avg_stay_days
FROM admissions
GROUP BY admission_type
ORDER BY total_admissions DESC;
""",
    "Top 10 primary diagnoses": """
SELECT
    primary_dx_title,
    COUNT(*) AS total_admissions
FROM admissions
WHERE primary_dx_title IS NOT NULL
GROUP BY primary_dx_title
ORDER BY total_admissions DESC
LIMIT 10;
""",
    "In-hospital deaths by age group": """
SELECT
    CASE
        WHEN age_at_admit < 40 THEN 'under 40'
        WHEN age_at_admit < 60 THEN '40 to 59'
        WHEN age_at_admit < 80 THEN '60 to 79'
        ELSE '80 and above'
    END AS age_group,
    COUNT(*) AS total_admissions,
    SUM(hospital_expire_flag) AS deaths,
    ROUND(100.0 * SUM(hospital_expire_flag) / COUNT(*), 1) AS death_pct
FROM admissions
GROUP BY age_group
ORDER BY MIN(age_at_admit);
""",
    "Stay length and diagnoses by insurance": """
SELECT
    insurance,
    COUNT(*) AS total_admissions,
    ROUND(AVG(los_days), 1) AS avg_stay_days,
    ROUND(AVG(num_diagnoses), 1) AS avg_diagnoses
FROM admissions
GROUP BY insurance
ORDER BY total_admissions DESC;
""",
}

for name, q in queries.items():
    cur.execute(q)
    print(name)
    print([c[0] for c in cur.description])
    for r in cur.fetchall():
        print(r)
    print()

checks = [
    """
SELECT 'patients' AS table_name, COUNT(*) AS total_rows FROM patients
UNION ALL
SELECT 'admissions', COUNT(*) FROM admissions
UNION ALL
SELECT 'diagnoses', COUNT(*) FROM diagnoses
""",
    """
SELECT COUNT(*) AS admissions_without_patient
FROM admissions a
LEFT JOIN patients p ON p.subject_id = a.subject_id
WHERE p.subject_id IS NULL
""",
    """
SELECT COUNT(*) AS negative_stay_rows
FROM admissions
WHERE los_days < 0
""",
    """
SELECT table_name, rows_read, rows_loaded, rows_rejected, run_time
FROM etl_log
ORDER BY id DESC
LIMIT 3
""",
]

Admissions and average stay by admission type
['admission_type', 'total_admissions', 'avg_stay_days']
('EW EMER.', 104, Decimal('7.3'))
('OBSERVATION ADMIT', 45, Decimal('8.2'))
('URGENT', 38, Decimal('9.9'))
('EU OBSERVATION', 30, Decimal('0.9'))
('SURGICAL SAME DAY ADMISSION', 18, Decimal('5.7'))
('DIRECT EMER.', 15, Decimal('9.4'))
('ELECTIVE', 13, Decimal('8.2'))
('DIRECT OBSERVATION', 7, Decimal('1.4'))
('AMBULATORY OBSERVATION', 5, Decimal('1.0'))

Top 10 primary diagnoses
['primary_dx_title', 'total_admissions']
('Coronary atherosclerosis of native coronary artery', 7)
('Acute kidney failure, unspecified', 7)
('Aortic valve disorders', 4)
('Non-ST elevation (NSTEMI) myocardial infarction', 4)
('Other postoperative infection', 4)
('Cerebral aneurysm, nonruptured', 4)
('Acute on chronic diastolic heart failure', 3)
('Encounter for antineoplastic chemotherapy', 3)
('Sepsis, unspecified organism', 3)
('Atrial fibrillation', 3)

In-hospital deaths by age group
['age_group', 'total_ad

In [20]:
for q in checks:
    cur.execute(q)
    print([c[0] for c in cur.description])
    for r in cur.fetchall():
        print(r)
    print()

cur.close()
conn.close()

['table_name', 'total_rows']
('patients', 100)
('admissions', 275)
('diagnoses', 4506)

['admissions_without_patient']
(0,)

['negative_stay_rows']
(0,)

['table_name', 'rows_read', 'rows_loaded', 'rows_rejected', 'run_time']
('diagnoses', 4506, 4506, 0, datetime.datetime(2026, 11, 1, 10, 50, 28, 375708))
('admissions', 275, 275, 0, datetime.datetime(2026, 11, 1, 10, 50, 28, 375708))
('patients', 100, 100, 0, datetime.datetime(2026, 11, 1, 10, 50, 28, 375708))

